<a href="https://colab.research.google.com/github/Meku-nint/C-/blob/main/modelCompression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install required libraries
!pip install transformers torchaudio datasets evaluate psutil jiwer torchcodec

In [ ]:
# Cell 2: Import necessary libraries
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import os
import time
import numpy as np
import psutil
from datasets import load_dataset
from evaluate import load


In [ ]:
# Cell 3: Load the Wav2Vec2 pretrained model
model_name = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

model.eval()
device = "cpu"
model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [ ]:
# Cell 4: Baseline measurements

# Model size
torch.save(model.state_dict(), "wav2vec2_baseline.pth")
model_size = os.path.getsize("wav2vec2_baseline.pth") / 1e6

# RAM usage
process = psutil.Process(os.getpid())
memory_usage = process.memory_info().rss / 1e6

# Dummy audio for inference timing
dummy_audio = np.random.randn(16000)
inputs = processor(dummy_audio, sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Inference time
start = time.time()
with torch.no_grad():
    _ = model(inputs["input_values"])
end = time.time()
inference_time = end - start

In [ ]:
# Cell 5: Evaluate baseline Word Error Rate (WER)
dataset = load_dataset("librispeech_asr", "clean", split="validation[:10]")  # Load first 10 samples
wer_metric = load("wer")

def evaluate_wer(model, dataset, max_samples=10):
    predictions, references = [], []
    # Ensure we don't try to access more samples than available in the dataset
    num_samples_to_evaluate = min(len(dataset), max_samples)
    for i in range(num_samples_to_evaluate):
        audio = dataset[i]["audio"]
        inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt")
        with torch.no_grad():
            logits = model(inputs.input_values).logits
        pred_ids = torch.argmax(logits, dim=-1)
        predicted_text = processor.decode(pred_ids[0])
        predictions.append(predicted_text.lower())
        references.append(dataset[i]["text"].lower())
    return wer_metric.compute(predictions=predictions, references=references)

baseline_wer = evaluate_wer(model, dataset)
baseline_wer_percent = baseline_wer * 100
word_accuracy_percent = (1 - baseline_wer) * 100
print(" BASELINE RESULTS")
print(f"Memory Usage     : {memory_usage:.2f} MB")
print(f"Inference Time   : {inference_time:.4f} sec")
print(f"Word Error Rate  : {baseline_wer_percent:.2f} %")
print(f"Word Accuracy    : {word_accuracy_percent:.2f} %")

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

 BASELINE RESULTS
Memory Usage     : 1509.95 MB
Inference Time   : 1.1168 sec
Word Error Rate  : 1.74 %
Word Accuracy    : 98.26 %


In [ ]:
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        # We prune entire rows (output features) instead of individual weights
        prune.ln_structured(module, name="weight", amount=0.2, n=1, dim=0)
        prune.remove(module, 'weight')
        print(f"Structuredly pruned 20% of neurons in: {name}")


NameError: name 'prune' is not defined

In [ ]:
# Cell 7: Measure pruned model and compare
torch.save(model.state_dict(), "wav2vec2_pruned.pth")
pruned_size = os.path.getsize("wav2vec2_pruned.pth") / 1e6
memory_usage_pruned = process.memory_info().rss / 1e6

# Inference time
start = time.time()
with torch.no_grad():
    _ = model(inputs["input_values"])
end = time.time()
inference_time_pruned = end - start

pruned_wer = evaluate_wer(model, dataset)
pruned_wer_percent = pruned_wer * 100
pruned_accuracy_percent = (1 - pruned_wer) * 100
print("===== PRUNED RESULTS =====")
print(f"Word Error Rate  : {pruned_wer_percent:.2f} %")
print(f"Word Accuracy    : {pruned_accuracy_percent:.2f} %")
print(f"Memory Usage     : {memory_usage_pruned:.2f} MB")
print(f"Inference Time   : {inference_time_pruned:.4f} sec")


In [ ]:
import torch
from torch.quantization import quantize_dynamic
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import time
import numpy as np
import psutil

# Load baseline model
model_name = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)
model.eval()
device = "cpu"
model.to(device)

# Dummy audio
dummy_audio = np.random.randn(16000)
inputs = processor(dummy_audio, sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Apply dynamic quantization on all Linear layers
quantized_model = quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

# Save quantized model
torch.save(quantized_model.state_dict(), "wav2vec2_quantized.pth")

# Measure model size
import os
quantized_size = os.path.getsize("wav2vec2_quantized.pth") / 1e6

# Warmup
for _ in range(3):
    with torch.no_grad():
        _ = quantized_model(inputs["input_values"])

# Measure inference time
start = time.time()
with torch.no_grad():
    _ = quantized_model(inputs["input_values"])
end = time.time()
inference_time_quant = end - start

# Measure RAM usage
process = psutil.Process(os.getpid())
memory_usage_quant = process.memory_info().rss / 1e6

print("===== QUANTIZED RESULTS =====")
print(f"Model Size       : {quantized_size:.2f} MB")
print(f"Memory Usage     : {memory_usage_quant:.2f} MB")
print(f"Inference Time   : {inference_time_quant:.4f} sec")
